In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

import s3fs
import configparser
from geopy.distance import distance
import geopandas as gpd

import echopype as ep

In [2]:
path_main = "/media/volume/shimada_202506_volume/integrated" # path to local integrated data
path_trawl = "/media/volume/shimada_202506_volume/trawl" # path to local trawl info
path_NASC_files = "agr230002-bucket01/prefect_sh2506_test/acoustics/NASC" # path to cloud acoustic data
cred_file = "/home/exouser/.config/rclone/rclone.conf"

In [3]:
file_NASC_all = "NASC_all.csv"
file_stratum_mean = "stratum_mean.csv"
file_NASC_all_grid = "NASC_all_griddify.geojson"
num_NASC_reprocess = 1

In [4]:
file_NASC_all = Path(path_main) / file_NASC_all
df_NASC_all = (
    pd.read_csv(file_NASC_all, index_col=0)
    if file_NASC_all.exists() else pd.DataFrame()
)
NASC_processed = (
    sorted(df_NASC_all["filename"].unique().tolist())
    if not df_NASC_all.empty else []
)

In [5]:
f"NASC files already processed: {NASC_processed}"

"NASC files already processed: ['NASC_20250613T114000.zarr', 'NASC_20250613T122000.zarr', 'NASC_20250613T130000.zarr', 'NASC_20250613T134000.zarr', 'NASC_20250613T142000.zarr', 'NASC_20250613T150000.zarr', 'NASC_20250613T154000.zarr', 'NASC_20250613T162000.zarr', 'NASC_20250613T170000.zarr', 'NASC_20250613T174000.zarr', 'NASC_20250613T182000.zarr', 'NASC_20250613T190000.zarr', 'NASC_20250613T194000.zarr', 'NASC_20250613T202000.zarr', 'NASC_20250613T210000.zarr', 'NASC_20250613T214000.zarr', 'NASC_20250613T222000.zarr', 'NASC_20250613T230000.zarr', 'NASC_20250613T234000.zarr', 'NASC_20250614T002000.zarr', 'NASC_20250614T114000.zarr', 'NASC_20250614T122000.zarr', 'NASC_20250614T130000.zarr', 'NASC_20250614T134000.zarr', 'NASC_20250614T142000.zarr', 'NASC_20250614T150000.zarr', 'NASC_20250614T154000.zarr', 'NASC_20250614T162000.zarr', 'NASC_20250614T170000.zarr', 'NASC_20250614T174000.zarr', 'NASC_20250614T182000.zarr', 'NASC_20250614T190000.zarr', 'NASC_20250614T194000.zarr', 'NASC_20250

In [6]:
NASC_to_reprocess = NASC_processed[-num_NASC_reprocess:]
NASC_processed = NASC_processed[:-num_NASC_reprocess]
f"NASC files to reprocess: {NASC_to_reprocess}"

"NASC files to reprocess: ['NASC_20250905T182000.zarr']"

In [7]:
df_NASC_all.empty

False

In [8]:
config = configparser.ConfigParser()
config.read(cred_file)
fs = s3fs.S3FileSystem(
    key=config["osn_sdsc_hake"]["access_key_id"],
    secret=config["osn_sdsc_hake"]["secret_access_key"],
    client_kwargs={"endpoint_url": config["osn_sdsc_hake"]["endpoint"]},
)

In [9]:
NASC_all = fs.glob(f"{path_NASC_files}/*.zarr")
NASC_all = sorted([Path(f).name for f in NASC_all])
f"All NASC files: {NASC_all}"

"All NASC files: ['NASC_20250613T114000.zarr', 'NASC_20250613T122000.zarr', 'NASC_20250613T130000.zarr', 'NASC_20250613T134000.zarr', 'NASC_20250613T142000.zarr', 'NASC_20250613T150000.zarr', 'NASC_20250613T154000.zarr', 'NASC_20250613T162000.zarr', 'NASC_20250613T170000.zarr', 'NASC_20250613T174000.zarr', 'NASC_20250613T182000.zarr', 'NASC_20250613T190000.zarr', 'NASC_20250613T194000.zarr', 'NASC_20250613T202000.zarr', 'NASC_20250613T210000.zarr', 'NASC_20250613T214000.zarr', 'NASC_20250613T222000.zarr', 'NASC_20250613T230000.zarr', 'NASC_20250613T234000.zarr', 'NASC_20250614T002000.zarr', 'NASC_20250614T114000.zarr', 'NASC_20250614T122000.zarr', 'NASC_20250614T130000.zarr', 'NASC_20250614T134000.zarr', 'NASC_20250614T142000.zarr', 'NASC_20250614T150000.zarr', 'NASC_20250614T154000.zarr', 'NASC_20250614T162000.zarr', 'NASC_20250614T170000.zarr', 'NASC_20250614T174000.zarr', 'NASC_20250614T182000.zarr', 'NASC_20250614T190000.zarr', 'NASC_20250614T194000.zarr', 'NASC_20250614T202000.zar

In [10]:
NASC_to_process = sorted(list(set(NASC_all).difference(set(NASC_processed))))

In [11]:
NASC_filenames = NASC_to_process

In [28]:
def task_combine_NASC_to_dataframe(
    fs: s3fs.S3FileSystem,
    path_NASC_files: str,
    NASC_filenames: list,
) -> list[pd.DataFrame, list]:
    """
    Combine multiple NASC datasets into a single DataFrame.

    Parameters
    ----------
    path_NASC_files : str
        Path to the directory containing NASC zarr files.
    NASC_filenames : list
        List of NASC zarr filenames to process.
        Not the full path, just the filenames.
    """
    df_NASC_list = []
    errors = []
    for nascf in NASC_filenames:
        try:
            mapper = fs.get_mapper(str(Path(path_NASC_files) / nascf))
            ds_NASC = xr.open_zarr(mapper, consolidated=True)
            ds_NASC = ep.consolidate.swap_dims_channel_frequency(ds_NASC)
            df_NASC = ds_NASC.sum("depth").sel(frequency_nominal=38000).to_dataframe()
            df_NASC["filename"] = nascf
            df_NASC_list.append(df_NASC)
        except Exception as e:
            errors.append(e)
            print(f"Error processing {nascf}: {e}")
            continue

    if len(df_NASC_list) == 0:
        return None, None
    else:
        return pd.concat(df_NASC_list, ignore_index=True), errors

In [29]:
df_NASC, errors = task_combine_NASC_to_dataframe(fs, path_NASC_files, NASC_to_process)

Error processing NASC_20250905T182000.zarr: Failed to decode variable 'ping_time': cannot reshape array of size 11 into shape (10,)


In [30]:
df_NASC

In [32]:
df_NASC_all = pd.concat([df_NASC_all, df_NASC], ignore_index=True)

In [33]:
df_stratum = pd.read_csv(Path(path_trawl) / file_stratum_mean, index_col=0)

In [35]:
from core import GRID_PARAMS
from grid import create_boundary_gdf, create_grid_from_bounds
from flows_biology import add_stratum

In [36]:
# Create boundary GeoDataFrame with UTM projection
gdf_boundary, gdf_boundary_utm, utm_num = create_boundary_gdf(
    bounds=GRID_PARAMS["bounds"], 
    projection=GRID_PARAMS["projection"]
)

In [57]:
def task_griddify_NASC(
    df_stratum: pd.DataFrame,
    df_NASC: pd.DataFrame,
    utm_num: int = utm_num,
    gdf_boundary_utm: gpd.GeoDataFrame = gdf_boundary_utm,
) -> gpd.GeoDataFrame:
    """
    Generate geodataframe with grid x/y containing NASC values.
    """
    # Convert to GeoDataFrame
    gdf_NASC = gpd.GeoDataFrame(
        data=df_NASC,
        geometry=gpd.points_from_xy(df_NASC["longitude"], df_NASC["latitude"]),
        crs=GRID_PARAMS["projection"],
    ).to_crs(f"epsg:{utm_num}")

    # Extract x y coordinate for pd.cut below
    gdf_NASC["utm_x"] = gdf_NASC["geometry"].x
    gdf_NASC["utm_y"] = gdf_NASC["geometry"].y

    # Get grid step sizes and boundary x/y
    x_step = distance(nautical=GRID_PARAMS["resolution"]["x_distance"]).meters
    y_step = distance(nautical=GRID_PARAMS["resolution"]["y_distance"]).meters
    xmin, ymin, xmax, ymax = gdf_boundary_utm.total_bounds

    # Bin longitude and latitude into grids
    gdf_NASC["grid_x"] = pd.cut(
        gdf_NASC["utm_x"],
        np.arange(xmin, xmax + x_step, x_step),
        right=False,
        labels=np.arange(1, len(np.arange(xmin, xmax + x_step, x_step))),
    )#.astype(int)

    # Bin the latitude data
    gdf_NASC["grid_y"] = pd.cut(
        gdf_NASC["utm_y"],
        np.arange(ymin, ymax + y_step, y_step),
        right=True,
        labels=np.arange(1, len(np.arange(ymin, ymax + y_step, y_step))),
    )#.astype(int)

    # Add stratum info
    gdf_NASC = add_stratum(gdf_NASC, df_stratum)

    return gdf_NASC

In [58]:
df_NASC

,NASC,frequency_nominal,latitude,longitude,ping_time,channel,filename
0,0.0,38000.0,32.672269,-117.751696,2025-06-13 11:57:37.500000000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr
1,0.0,38000.0,32.674293,-117.761954,2025-06-13 12:08:45.000000256,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr
2,0.0,38000.0,32.673825,-117.769001,2025-06-13 12:17:00.000000000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr
3,0.0,38000.0,32.670349,-117.771896,2025-06-13 12:22:32.500000256,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr
4,0.0,38000.0,32.661586,-117.769040,2025-06-13 12:27:15.000000000,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr
...,...,...,...,...,...,...,...
3609,0.0,38000.0,47.536662,-124.995316,2025-09-05 18:40:34.999999744,WBT 400143-15 ES38B_ES,NASC_20250905T182000.zarr
3610,0.0,38000.0,47.537071,-124.982997,2025-09-05 18:43:42.500000,WBT 400143-15 ES38B_ES,NASC_20250905T182000.zarr
3611,0.0,38000.0,47.537817,-124.970709,2025-09-05 18:46:50,WBT 400143-15 ES38B_ES,NASC_20250905T182000.zarr
3612,0.0,38000.0,47.538533,-124.958495,2025-09-05 18:49:55,WBT 400143-15 ES38B_ES,NASC_20250905T182000.zarr


In [60]:
gdf_NASC = gpd.GeoDataFrame(
    data=df_NASC,
    geometry=gpd.points_from_xy(df_NASC["longitude"], df_NASC["latitude"]),
    crs=GRID_PARAMS["projection"],
).to_crs(f"epsg:{utm_num}")

# Extract x y coordinate for pd.cut below
gdf_NASC["utm_x"] = gdf_NASC["geometry"].x
gdf_NASC["utm_y"] = gdf_NASC["geometry"].y

# Get grid step sizes and boundary x/y
x_step = distance(nautical=GRID_PARAMS["resolution"]["x_distance"]).meters
y_step = distance(nautical=GRID_PARAMS["resolution"]["y_distance"]).meters
xmin, ymin, xmax, ymax = gdf_boundary_utm.total_bounds

# Bin longitude and latitude into grids
gdf_NASC["grid_x"] = pd.cut(
    gdf_NASC["utm_x"],
    np.arange(xmin, xmax + x_step, x_step),
    right=False,
    labels=np.arange(1, len(np.arange(xmin, xmax + x_step, x_step))),
)#.astype(int)

# Bin the latitude data
gdf_NASC["grid_y"] = pd.cut(
    gdf_NASC["utm_y"],
    np.arange(ymin, ymax + y_step, y_step),
    right=True,
    labels=np.arange(1, len(np.arange(ymin, ymax + y_step, y_step))),
)#.astype(int)


In [61]:
# Create latitude bins from the stratum definitions
lat_bins = [-90.0] + df_stratum["latitude_northern_limit"].tolist() + [90.0]
lat_labels = df_stratum["stratum"].tolist() + [max(df_stratum["stratum"]) + 1]

In [65]:
df_NASC["stratum"] = pd.cut(
        df_NASC["latitude"],
        bins=lat_bins,
        labels=lat_labels,
        include_lowest=True
    )

In [71]:
type(df_NASC)

pandas.core.frame.DataFrame

In [72]:
file_NASC_all_grid = "NASC_all_griddify.geojson"
file_stratum_mean = "stratum_mean.csv"
file_grid_cells = "grid_cells.geojson"

In [75]:
gdf_NASC = gpd.read_file(Path(path_main) / file_NASC_all_grid)

In [76]:
# Create gdf_grid_cells
gdf_grid_cells, _, _ = create_grid_from_bounds(
    bounds=GRID_PARAMS["bounds"], 
    resolution=GRID_PARAMS["resolution"],
    projection=GRID_PARAMS["projection"],
    coastline_resolution="10m",
    area_threshold=5
)
gdf_grid_cells.set_index(["grid_x", "grid_y"], inplace=True)

In [77]:
# Merge on stratum 
gdf_NASC = gdf_NASC.merge(
    df_stratum[["stratum", "sigma_bs_mean", "weight_mean"]],
    on="stratum",  # merge on stratum 
    how="left"
)

# Compute stratified number density from NASC
gdf_NASC["number_density"] = gdf_NASC["NASC"] / gdf_NASC["sigma_bs_mean"]

# Compute biomass density from number density
gdf_NASC["biomass_density"] = gdf_NASC["number_density"] * gdf_NASC["weight_mean"]

# Merge with grid cells
gdf_grid_cells = pd.merge(
    gdf_grid_cells,
    gdf_NASC.groupby(["grid_x", "grid_y"])["NASC"].mean(),
    on=["grid_x", "grid_y"],
    how="left"
)
gdf_grid_cells = pd.merge(
    gdf_grid_cells,
    gdf_NASC.groupby(["grid_x", "grid_y"])["number_density"].mean(),
    on=["grid_x", "grid_y"],
    how="left"
)
gdf_grid_cells = pd.merge(
    gdf_grid_cells,
    gdf_NASC.groupby(["grid_x", "grid_y"])["biomass_density"].mean(),
    on=["grid_x", "grid_y"],
    how="left"
)

# Compute abundance and biomass
gdf_grid_cells["abundance"] = gdf_grid_cells["number_density"] * gdf_grid_cells["area"]
gdf_grid_cells["biomass"] = gdf_grid_cells["biomass_density"] * gdf_grid_cells["area"]


In [78]:
gdf_grid_cells

,,geometry,area,NASC,number_density,biomass_density,abundance,biomass
grid_x,grid_y,,,,,,,
38,1,"POLYGON ((-117.25452 32.76713, -117.25388 32.7...",491.943240,NaN,NaN,NaN,NaN,NaN
37,1,"POLYGON ((-117.4894 32.78924, -117.49448 32.75...",625.000000,NaN,NaN,NaN,NaN,NaN
36,1,"POLYGON ((-118.44992 32.81882, -118.43837 32.8...",610.460377,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
35,1,"POLYGON ((-118.51361 32.88037, -118.5086 32.87...",620.304729,16.659054,168006.540750,18408.689904,1.042153e+08,1.141900e+07
34,1,"POLYGON ((-118.94753 32.9164, -118.96635 32.75...",625.000000,2.099384,21172.289714,2319.874656,1.323268e+07,1.449922e+06
...,...,...,...,...,...,...,...,...
5,53,"POLYGON ((-135.24446 54.8394, -135.18116 54.42...",625.000000,NaN,NaN,NaN,NaN,NaN
6,53,"POLYGON ((-134.52703 54.87429, -134.47091 54.4...",625.000000,NaN,NaN,NaN,NaN,NaN
5,54,"POLYGON ((-135.24446 54.8394, -135.25 54.83909...",625.000000,NaN,NaN,NaN,NaN,NaN


In [79]:
import cartopy.feature as cfeature
from geopy.distance import distance
import pandas as pd
import numpy as np
import geopandas as gpd
import shapely
import shapely.geometry as sg

In [80]:
# Plotting setup
import geoviews as gv
import holoviews as hv
import panel as pn
from bokeh.models import HoverTool
from bokeh.palettes import Viridis256
from holoviews.streams import RangeXY

In [86]:
from bokeh.themes.theme import Theme
import geoviews.tile_sources as gvts

# Plot theme
THEME = Theme(
    json={
        "attrs": {
            "Title": {
                "align": "left",
                "text_font_size": "20px",
                "text_color": "black",
            },
            "Axis": {
                "axis_label_text_font_style": "bold",
                "axis_label_text_color": "black",
                "axis_label_text_font_size": "18px",
                "major_label_text_font_size": "15px",
                "major_label_text_color": "black",
            },
            "ColorBar": {
                "title_text_font_style": "normal",
                "title_text_font_size": "18px",
                "title_text_color": "black",
                "major_label_text_font_size": "16px",
                "major_label_text_color": "black",
            },
            "Legend": {
                "title_text_font_style": "bold",
                "title_text_font_size": "16px",
                "title_text_color": "black",
            }
        }
    }
)


# Variable display map
VARIABLE_MAP = {
    "NASC": "NASC",
    "Abundance": "abundance",
    "Biomass": "biomass",
    "Number density": "number_density",
    "Biomass density": "biomass_density"
}

# Colorbar title name map
COLORBAR_NAME_MAP = {
    "NASC": r'$$\sum \mathrm{NASC}$$',
    "abundance": r"$$\sum N$$",
    "biomass": r"$$\sum \mathrm{kg}$$",
    "number_density": r"$$\overline{N~\mathrm{nmi^2}}$$",
    "biomass_density": r"$$\overline{\mathrm{kg~nmi^2}}$$",
}

# Variable units map
VARIABLE_UNITS_MAP = {
    "NASC": "m² nmi⁻²",
    "abundance": "N",
    "biomass": "kg",
    "number_density": "N nmi⁻²",
    "biomass_density": "kg nmi⁻²",
}

# Tile source map
TILE_MAP = {
    "ESRI NatGeo": gvts.EsriNatGeo,
    "ESRI Imagery": gvts.EsriImagery,
    "OpenStreetMap": gvts.OSM,
    "Carto Light": gvts.CartoLight,
    "Carto Dark": gvts.CartoDark,
    "OpenTopoMap": gvts.OpenTopoMap,
}

In [87]:

# Helper functions
def get_linestring(
    data_gdf: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """
    Extract the linestrings from the transect data
    """

    # Create lines
    line = sg.LineString(data_gdf.geometry.tolist())

    # Create an updated GeoDataFrame
    line_gdf = gpd.GeoDataFrame(geometry=[line], 
                                crs=data_gdf.crs)

    # Return
    return line_gdf

def clean_cells(
    grid_gdf: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """
    Clean POLYGON/MULTIPOLYGON objects
    """

    # Remove any POINT values
    poly_gdf = grid_gdf[
        grid_gdf.geometry.apply(lambda x: isinstance(x, (shapely.geometry.Polygon, shapely.geometry.MultiPolygon)))
    ]

    return poly_gdf

# Define widgets
VARIABLE_SELECTOR = pn.widgets.Select(name="Select variable", 
                                      options=list(VARIABLE_MAP.keys()),
                                      value="NASC")

TILE_SELECTOR = pn.widgets.Select(name="Basemap tile source",
                                  options=list(TILE_MAP.keys()),
                                  value="ESRI Imagery")

LINE_TOGGLE = pn.widgets.Checkbox(name="Show transect path(s)", 
                                  value=False)

# Plotting function
def plot_grid(
    cells_gdf: gpd.GeoDataFrame,
    # transect_gdf: gpd.GeoDataFrame,
) -> None:
    """
    Plotting function
    """

    # Create copy
    cells_gdf = cells_gdf.copy()

    # Create linestrings
    # lines_gdf = get_linestring(transect_gdf)

    # Initialize extensions
    hv.extension("bokeh", enable_mathjax=True)
    hv.renderer("bokeh").theme = THEME
    pn.extension()

    # Create palette
    palette = list(Viridis256)

    # Clean 'cells_gdf'
    cells_gdf = clean_cells(cells_gdf)

    # Create plotting function
    @pn.depends(VARIABLE_SELECTOR, TILE_SELECTOR, LINE_TOGGLE)
    def render_polygons(
        variable: str,
        map_tile: gv.element.geo.WMTS,
        line_draw: bool,
    ) -> None:
        """
        Panel plotting function
        """      
    
        # Get column name
        var = VARIABLE_MAP.get(variable, variable)

        # Get the variable units
        var_units = VARIABLE_UNITS_MAP.get(var, var)

        # Get colorbar name/title
        colorbar_title = COLORBAR_NAME_MAP.get(var, var)

        # Get the base tilemap
        tile = TILE_MAP.get(map_tile, map_tile)

        # Get data limits
        data_min, data_max = np.nanmin(cells_gdf[var]), np.nanmax(cells_gdf[var]) * 1.01
        
        # Add label column
        cells_gdf.loc[:, f"{var}_label"] = cells_gdf.loc[:, var].apply(
            lambda v: "Unsampled" if pd.isna(v) else f"{v:.2e} {var_units}"
        )
        
        # Update the hover tooltipe
        hover_tool = HoverTool(
            tooltips=[
                (f"{variable}", f"@{var}_label"),
                ("Area", "@area nmi²"),
            ]
        )

        # Begin plotting
        cells = gv.Polygons(
            cells_gdf, 
            vdims=[var, f"{var}_label", "area"]
        ).opts(
            # Dimensions
            width=900,
            height=800,
            # Polygon colors
            colorbar=True,
            cmap=palette,
            clim=(data_min, data_max),
            clipping_colors={
                "NaN": "#c1c1c1",
            },
            colorbar_opts={
                "title": colorbar_title,
            },
            line_color="black",
            alpha=0.8,
            labelled=[variable],
            # Axis labels
            xlabel="Longitude (\u00B0E)",
            ylabel="Latitude (\u00B0N)",
            # Plot title
            title=f"{variable}",
            # Hover tooltip
            tools=[hover_tool],
            
        )

        # Combine with tilemap
        overlay = tile * cells

        # # Draw transect track?
        # if line_draw:
        #     line_layer = gv.Path(lines_gdf).opts(
        #         color="black", line_width=2.
        #     )
        #     # ---- Add to the overlay layers
        #     overlay = overlay * line_layer

        return overlay

    # Format panel
    layout = pn.Column(VARIABLE_SELECTOR, TILE_SELECTOR, LINE_TOGGLE, render_polygons)
    # return layout
    layout.show()

In [88]:
plot_grid(cells_gdf=gdf_grid_cells)

/home/exouser/miniforge3/envs/echodataflow_20250609/lib/python3.12/site-packages/holoviews/util/__init__.py:754: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("mathjax")


/tmp/ipykernel_115348/1827169979.py:63: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension()


Launching server at http://localhost:45747
